In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
import requests
import base64
# Get the file listing from the GitHub API
api_url = "https://api.github.com/repos/marcomorucci/Clustering-Constitutions/contents/constitutions"
response = requests.get(api_url)
files = response.json()
constitutions = {}
for file_info in files:
    if file_info["name"].endswith(".txt"):
        # Fetch each file's raw content
        raw_url = file_info["download_url"]
        text = requests.get(raw_url).text
        country = file_info["name"].replace(".txt", "")
        constitutions[country] = text
print(f"Loaded {len(constitutions)} constitutions")

Loaded 192 constitutions


In [3]:
num_docs = len(constitutions)
print(f"1. Number of documents: {num_docs}")

1. Number of documents: 192


In [4]:
# Character-level lengths
char_lengths = {country: len(text) for country, text in constitutions.items()}
lengths_arr = np.array(list(char_lengths.values()))

print(f"2. Average length (chars): {lengths_arr.mean():,.0f}")
print(f"3. Max length (chars):     {lengths_arr.max():,} — {max(char_lengths, key=char_lengths.get)}")
print(f"   Min length (chars):     {lengths_arr.min():,} — {min(char_lengths, key=char_lengths.get)}")

2. Average length (chars): 138,099
3. Max length (chars):     616,928 — India_2012
   Min length (chars):     18,926 — Libya_2011


In [5]:
# Word-level lengths (often more interpretable for NLP)
word_lengths = {c: len(t.split()) for c, t in constitutions.items()}
word_arr = np.array(list(word_lengths.values()))
print(f"\n   Average length (words): {word_arr.mean():,.0f}")
print(f"   Max length (words):     {word_arr.max():,} — {max(word_lengths, key=word_lengths.get)}")
print(f"   Min length (words):     {word_arr.min():,} — {min(word_lengths, key=word_lengths.get)}")



   Average length (words): 22,173
   Max length (words):     102,720 — India_2012
   Min length (words):     2,939 — Libya_2011


In [6]:
 
# ── 4: OHCO — parse into a structured DataFrame ──────────────────────────────
#
# OHCO levels for this corpus:
#   Document > Chapter/Part/Title > Article > Paragraph > Sentence > Token
#
# The most reliable cross-corpus unit is the Article.
# Top-level divisions vary by country ("Chapter", "Title", "Part", "Section").

def parse_constitution(country, text):
    """
    Parse a constitution into (country, division, article, para, sentence) rows.
    Returns a list of dicts — one per sentence.
    """
    rows = []

    # Split into top-level divisions (Chapter / Title / Part / Section + roman/arabic number)
    division_pattern = re.compile(
        r'^(Chapter|Title|Part|Section|CHAPTER|TITLE|PART|SECTION)\s+[\dIVXivx]+',
        re.MULTILINE
    )
    division_splits = division_pattern.split(text)

    # Simpler approach: walk through articles sequentially, tracking last seen division
    current_division = "Preamble"
    article_pattern  = re.compile(r'Article\s+(\d+)', re.IGNORECASE)
    div_pattern      = re.compile(
        r'(Chapter|Title|Part|Section)\s+([\dIVXivx]+)', re.IGNORECASE
    )

    # Split text on article boundaries
    article_splits = article_pattern.split(text)
    # article_splits = [pre_text, art_num, art_body, art_num, art_body, ...]

    # Handle preamble (text before first Article)
    preamble_text = article_splits[0]
    for div_match in div_pattern.finditer(preamble_text):
        current_division = f"{div_match.group(1).title()} {div_match.group(2)}"

    # Process each article
    i = 1
    while i < len(article_splits) - 1:
        art_num  = article_splits[i]
        art_body = article_splits[i + 1]
        i += 2

        # Check if a new division heading appears before this article
        for div_match in div_pattern.finditer(art_body):
            current_division = f"{div_match.group(1).title()} {div_match.group(2)}"

        # Split article body into paragraphs (double newline or numbered sub-items)
        paragraphs = [p.strip() for p in re.split(r'\n{2,}', art_body) if p.strip()]

        for para_num, para in enumerate(paragraphs, 1):
            # Split paragraph into sentences (naive but workable)
            sentences = re.split(r'(?<=[.!?])\s+', para)
            for sent_num, sent in enumerate(sentences, 1):
                sent = sent.strip()
                if not sent:
                    continue
                rows.append({
                    'country':   country,
                    'division':  current_division,
                    'article':   int(art_num),
                    'paragraph': para_num,
                    'sentence':  sent_num,
                    'text':      sent,
                    'n_tokens':  len(sent.split()),
                })
    return rows

# Build the OHCO table (takes ~30–60s for all 196 constitutions)
all_rows = []
for country, text in constitutions.items():
    all_rows.extend(parse_constitution(country, text))

ohco_df = pd.DataFrame(all_rows)
ohco_df = ohco_df.set_index(['country', 'division', 'article', 'paragraph', 'sentence'])

print("\n4. OHCO structure:")
print(f"   Total sentences : {len(ohco_df):,}")
print(f"   Total tokens    : {ohco_df['n_tokens'].sum():,}")
print(f"   Unique divisions seen (top 10 labels):")
print(ohco_df.reset_index()['division'].value_counts().head(10))
print("\nSample rows:")
print(ohco_df.head(10))


4. OHCO structure:
   Total sentences : 195,615
   Total tokens    : 2,856,636
   Unique divisions seen (top 10 labels):
division
Part i         20473
Chapter II     13658
Chapter I      11868
Chapter III     8635
Preamble        7024
Chapter IV      6190
Chapter V       4993
Chapter 2       4885
Chapter VI      4067
Part 2          3852
Name: count, dtype: int64

Sample rows:
                                                                                                    text  \
country          division  article paragraph sentence                                                      
Afghanistan_2004 Chapter I 1       1         1                                                     Share   
                                   2         1         Afghanistan shall be an Islamic Republic, inde...   
                           2       1         1                                                     Share   
                                   2         1         The sacred religion of I

In [8]:
import json

with open("constitutions.json", "w", encoding="utf-8") as file:
    json.dump(constitutions, file, indent=4)